In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")
print(f"Pandas version: {pd.__version__}")

Matplotlib is building the font cache; this may take a moment.


Libraries loaded successfully!
Pandas version: 3.0.3


In [3]:
df = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    na_values=['?']
)

# Combine Date and Time into one DateTime column
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)

# Drop the original Date and Time columns
df = df.drop(columns=['Date', 'Time'])

print(f"Dataset loaded! Shape: {df.shape}")
print(f"\nDate range: {df['DateTime'].min()} to {df['DateTime'].max()}")
df.head()

Dataset loaded! Shape: (2075259, 8)

Date range: 2006-12-16 17:24:00 to 2010-11-26 21:02:00


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


In [4]:
print(df.columns.tolist())
print(f"\n")
df.info()

['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'DateTime']


<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 8 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Global_active_power    float64       
 1   Global_reactive_power  float64       
 2   Voltage                float64       
 3   Global_intensity       float64       
 4   Sub_metering_1         float64       
 5   Sub_metering_2         float64       
 6   Sub_metering_3         float64       
 7   DateTime               datetime64[us]
dtypes: datetime64[us](1), float64(7)
memory usage: 126.7 MB


In [5]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({
    'Missing Values': missing,
    'Percentage': missing_pct
})

,Missing Values,Percentage
Global_active_power,25979,1.25
Global_reactive_power,25979,1.25
Voltage,25979,1.25
Global_intensity,25979,1.25
Sub_metering_1,25979,1.25
Sub_metering_2,25979,1.25
Sub_metering_3,25979,1.25
DateTime,0,0.00


In [6]:
print(f"Rows before dropping missing: {len(df)}")
df = df.dropna()
print(f"Rows after dropping missing: {len(df)}")
print(f"Rows removed: {25979}")

Rows before dropping missing: 2075259
Rows after dropping missing: 2049280
Rows removed: 25979


In [7]:
# Set DateTime as index
df = df.set_index('DateTime')

# 1. Hourly average by hour of day (average across all days)
hourly_pattern = df['Global_active_power'].groupby(df.index.hour).mean().reset_index()
hourly_pattern.columns = ['Hour', 'Global_active_power']

# 2. Daily resampling
daily = df['Global_active_power'].resample('D').mean().reset_index()
daily.columns = ['Date', 'Global_active_power']

# 3. Monthly resampling
monthly = df['Global_active_power'].resample('ME').mean().reset_index()
monthly.columns = ['Month', 'Global_active_power']

print(f"Hourly pattern: {hourly_pattern.shape} — 24 hours")
print(f"Daily data: {daily.shape} — one row per day")
print(f"Monthly data: {monthly.shape} — one row per month")

Hourly pattern: (24, 2) — 24 hours
Daily data: (1442, 2) — one row per day
Monthly data: (48, 2) — one row per month


In [8]:
fig = px.line(
    hourly_pattern,
    x='Hour',
    y='Global_active_power',
    title='Average Electricity Consumption by Hour of Day',
    labels={'Hour': 'Hour of Day', 'Global_active_power': 'Avg Power (kW)'},
    markers=True
)

fig.update_layout(
    xaxis=dict(tickmode='linear', tick0=0, dtick=1),
    height=400
)

fig.show()

In [9]:
df['Hour'] = df.index.hour
df['DayType'] = df.index.dayofweek.map(
    lambda x: 'Weekend' if x >= 5 else 'Weekday'
)

hourly_split = df.groupby(['DayType', 'Hour'])['Global_active_power'].mean().reset_index()

fig = px.line(
    hourly_split,
    x='Hour',
    y='Global_active_power',
    color='DayType',
    title='Average Electricity Consumption by Hour — Weekday vs Weekend',
    labels={'Hour': 'Hour of Day', 'Global_active_power': 'Avg Power (kW)'},
    markers=True
)
fig.show()

In [10]:
fig = px.line(
    daily,
    x='Date',
    y='Global_active_power',
    title='Daily Average Electricity Consumption (2006-2010)',
    labels={'Date': 'Date', 'Global_active_power': 'Avg Power (kW)'},
)

fig.update_layout(height=400)
fig.show()

In [11]:
fig = px.line(
    monthly,
    x='Month',
    y='Global_active_power',
    title='Monthly Average Electricity Consumption (2006-2010)',
    labels={'Month': 'Month', 'Global_active_power': 'Avg Power (kW)'},
    markers=True
)

fig.update_layout(height=400)
fig.show()

In [12]:
# Check how many days of data we have per month
daily['YearMonth'] = daily['Date'].dt.to_period('M').astype(str)
days_per_month = daily.groupby('YearMonth')['Date'].count().reset_index()
days_per_month.columns = ['YearMonth', 'DaysOfData']

# Show the first and last few months
print("First months:")
print(days_per_month.head(10))
print("\nLast months:")
print(days_per_month.tail(5))

First months:
  YearMonth  DaysOfData
0   2006-12          16
1   2007-01          31
2   2007-02          28
3   2007-03          31
4   2007-04          30
5   2007-05          31
6   2007-06          30
7   2007-07          31
8   2007-08          31
9   2007-09          30

Last months:
   YearMonth  DaysOfData
43   2010-07          31
44   2010-08          31
45   2010-09          30
46   2010-10          31
47   2010-11          26


In [13]:
# Remove incomplete months
monthly_clean = monthly[
    ~monthly['Month'].dt.to_period('M').astype(str).isin(['2006-12', '2010-11'])
]

fig = px.line(
    monthly_clean,
    x='Month',
    y='Global_active_power',
    title='Monthly Average Electricity Consumption (excluding incomplete months)',
    labels={'Month': 'Month', 'Global_active_power': 'Avg Power (kW)'},
    markers=True
)

fig.update_layout(height=400)
fig.show()

In [14]:
# Day of week pattern
df['DayOfWeek'] = df.index.dayofweek
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 
             4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df['DayName'] = df['DayOfWeek'].map(day_names)

day_pattern = df.groupby('DayOfWeek')['Global_active_power'].mean().reset_index()
day_pattern['DayName'] = day_pattern['DayOfWeek'].map(day_names)

fig = px.bar(
    day_pattern,
    x='DayName',
    y='Global_active_power',
    title='Average Electricity Consumption by Day of Week',
    labels={'DayName': '', 'Global_active_power': 'Avg Power (kW)'},
    color='Global_active_power',
    color_continuous_scale='Blues',
    category_orders={'DayName': ['Monday', 'Tuesday', 'Wednesday', 
                                  'Thursday', 'Friday', 'Saturday', 'Sunday']}
)

fig.update_layout(height=400, coloraxis_showscale=False)
fig.show()

In [15]:
# Sub-metering breakdown
submeter_avg = {
    'Kitchen': df['Sub_metering_1'].mean(),
    'Laundry Room': df['Sub_metering_2'].mean(),
    'Water Heater & AC': df['Sub_metering_3'].mean(),
    'Unmetered': (df['Global_active_power'] * 1000/60 - 
                  df['Sub_metering_1'] - 
                  df['Sub_metering_2'] - 
                  df['Sub_metering_3']).mean()
}

submeter_df = pd.DataFrame({
    'Category': list(submeter_avg.keys()),
    'Avg_Consumption': list(submeter_avg.values())
})

fig = px.pie(
    submeter_df,
    names='Category',
    values='Avg_Consumption',
    title='Average Electricity Consumption by Category',
    color_discrete_sequence=['#4299e1', '#48bb78', '#ed8936', '#fc8181']
)

fig.update_layout(height=400)
fig.show()

In [17]:
fig = px.bar(
    submeter_df.sort_values('Avg_Consumption', ascending=True),
    x='Avg_Consumption',
    y='Category',
    title='Where Does the Electricity Go? Average Consumption by Category',
    labels={'Avg_Consumption': 'Avg Consumption (Wh/min)', 'Category': ''},
    color='Category',
    color_discrete_map={
        'Unmetered': '#fc8181',
        'Water Heater & AC': '#ed8936',
        'Kitchen': '#4299e1',
        'Laundry Room': '#48bb78'
    },
    orientation='h',
    text='Avg_Consumption'
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    height=350,
    showlegend=False,
    xaxis=dict(showticklabels=False)
)
fig.show()

In [23]:
import numpy as np
from scipy import stats

# Calculate mean and std of daily consumption
mu = daily['Global_active_power'].mean()
std = daily['Global_active_power'].std()

# Generate normal distribution curve
x = np.linspace(daily['Global_active_power'].min(), daily['Global_active_power'].max(), 200)
y = stats.norm.pdf(x, mu, std)

# Scale y to match histogram counts
bin_width = (daily['Global_active_power'].max() - daily['Global_active_power'].min()) / 50
y_scaled = y * len(daily) * bin_width

fig = px.histogram(
    daily,
    x='Global_active_power',
    nbins=50,
    title='Distribution of Daily Average Electricity Consumption',
    labels={'Global_active_power': 'Daily Avg Power (kW)'},
    color_discrete_sequence=['#4299e1']
)

fig.add_scatter(
    x=x,
    y=y_scaled,
    mode='lines',
    name='Normal Distribution',
    line=dict(color='red', width=2)
)

fig.update_layout(
    height=400,
    yaxis=dict(showticklabels=False, showgrid=False)
)

fig.show()

In [26]:
weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
hourly_weekdays = hourly_by_day[hourly_by_day['DayName'].isin(weekdays)]

fig = px.line(
    hourly_weekdays,
    x='Hour',
    y='Global_active_power',
    color='DayName',
    title='Average Hourly Consumption by Weekday',
    labels={'Hour': 'Hour of Day', 'Global_active_power': 'Avg Power (kW)'},
    markers=False,
    category_orders={'DayName': weekdays}
)

fig.update_layout(height=500)
fig.show()

In [27]:
# Save all pre-computed data for the backend
daily.to_csv('daily_consumption.csv', index=False)
monthly_clean.to_csv('monthly_consumption.csv', index=False)
hourly_pattern.to_csv('hourly_pattern.csv', index=False)
hourly_split.to_csv('hourly_weekday_weekend.csv', index=False)
day_pattern.to_csv('day_of_week.csv', index=False)
submeter_df.to_csv('submeter_breakdown.csv', index=False)
hourly_weekdays.to_csv('hourly_by_weekday.csv', index=False)

# Distribution data
daily[['Global_active_power']].to_csv('daily_distribution.csv', index=False)

print("All files saved!")

All files saved!
